# Stage 1 — Optuna Hyperparameter Search

This notebook runs an Optuna hyperparameter search for any one of the
9 scratch models defined in **`models_six.py`**, using the helpers in
**`optuna_utils.py`**.

**Workflow**
1. Load CATSA data (train / val split)
2. Build an Optuna objective via `make_objective()`
3. Run the study with `run_study()`
4. Inspect results with `report_study()` and visualise with Optuna plots
5. Save the best HP dict for Stage 2 full training

In [ ]:
import sys, json
from pathlib import Path

# ── add Temp dir to path so models_six & optuna_utils are importable ─────────
TEMP_DIR = Path('/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/Temp')
sys.path.insert(0, str(TEMP_DIR))

import numpy as np
import pandas as pd
import optuna
import torch
import matplotlib.pyplot as plt

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = torch.cuda.is_available()
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}  |  bf16: {USE_BF16}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
#
# Change MODEL_NAME to any of:
#   'TimeMixerPP' | 'Medformer' | 'TSLANet'  | 'ModernTCN' | 'CrossGNN'
#   'TimesNet'    | 'Mamba2'    | 'iTransformer' | 'PatchTST'
#
MODEL_NAME = 'TimeMixerPP'
N_TRIALS   = 50          # number of Optuna trials
N_EPOCHS   = 30          # max epochs per trial
PATIENCE   = 7           # early-stopping patience within each trial
VAL_RATIO  = 0.10        # fraction of subjects held out for validation
SEED       = 42

TASKS        = ['Baseline', 'Logic', 'Nback', 'Stroop', 'Sudoku']
STRESS_TASKS = {'Logic', 'Nback', 'Stroop', 'Sudoku'}
FS           = 4          # Temp / EDA sampling rate (Hz)
SIGNAL       = 'TEMP'     # 'TEMP' or 'EDA'

CATSA_ROOT = Path('/home/binghin2/Myproject/Dataset/CATSA')
SAVE_DIR   = TEMP_DIR / 'Stage1_Optuna_Results'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

W          = FS * 60      # window size = 240 samples
S          = FS * 10      # stride      =  40 samples
N_CHANNELS = 1

print(f'Model: {MODEL_NAME}  |  Trials: {N_TRIALS}  |  W={W}  S={S}')

In [ ]:
# ── Data utilities (same as Temp_TenModels.ipynb) ─────────────────────────────
def discover_subjects(root):
    out = []
    for d in sorted(root.glob('Sub*'), key=lambda p: int(p.name[3:])):
        if d.is_dir() and all((d / t / f'{SIGNAL}.csv').exists() for t in TASKS):
            out.append(d.name)
    return out

def read_signal(path):
    df = pd.read_csv(path)
    nc = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    v  = df[nc[0]].to_numpy(np.float32) if nc else pd.to_numeric(df.iloc[:, 0], errors='coerce').to_numpy(np.float32)
    return v[~np.isnan(v)]

def make_windows(sig, W, S):
    if len(sig) < W: return np.empty((0, 1, W), np.float32)
    return np.asarray([sig[i:i+W] for i in range(0, len(sig)-W+1, S)], np.float32)[:, None, :]

def build_catsa_arrays(subjects, W, S):
    xs, ys = [], []
    for sub in subjects:
        path  = CATSA_ROOT / sub
        all_v = np.concatenate([read_signal(path / t / f'{SIGNAL}.csv') for t in TASKS])
        mu    = float(np.mean(all_v)); sigma = float(np.std(all_v)) + 1e-8
        for task in TASKS:
            norm = (read_signal(path / task / f'{SIGNAL}.csv') - mu) / sigma
            w    = make_windows(norm, W, S)
            if len(w): xs.append(w); ys.append(np.full(len(w), int(task in STRESS_TASKS), np.int64))
    return np.concatenate(xs), np.concatenate(ys)

print('Data utilities ready.')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

subjects = discover_subjects(CATSA_ROOT)
print(f'Total CATSA subjects: {len(subjects)}')

rng        = np.random.default_rng(SEED)
idx        = rng.permutation(len(subjects))
n_val      = max(4, int(len(subjects) * VAL_RATIO))
val_subs   = [subjects[i] for i in sorted(idx[:n_val])]
train_subs = [subjects[i] for i in sorted(idx[n_val:])]
print(f'Train: {len(train_subs)} subjects | Val: {len(val_subs)} subjects')

print('Building windows…')
x_train, y_train = build_catsa_arrays(train_subs, W, S)
x_val,   y_val   = build_catsa_arrays(val_subs,   W, S)
print(f'Train: {x_train.shape}  Val: {x_val.shape}')

pos   = float(y_train.sum())
ALPHA = float((len(y_train) - pos) / len(y_train))
print(f'Stress: {pos:.0f}/{len(y_train)} ({pos/len(y_train)*100:.1f}%)  FocalLoss alpha={ALPHA:.3f}')

In [ ]:
# ── Build Optuna objective ────────────────────────────────────────────────────
from optuna_utils import make_objective, run_study, report_study

objective = make_objective(
    model_name = MODEL_NAME,
    x_tr       = x_train,
    y_tr       = y_train,
    x_va       = x_val,
    y_va       = y_val,
    W          = W,
    n_channels = N_CHANNELS,
    device     = DEVICE,
    alpha      = ALPHA,
    n_epochs   = N_EPOCHS,
    patience   = PATIENCE,
    use_bf16   = USE_BF16,
)
print('Objective built — ready to run study.')

In [ ]:
# ── Run Optuna study ──────────────────────────────────────────────────────────
# Results are persisted to an SQLite DB so you can resume after interruption.
DB_PATH    = f'sqlite:///{SAVE_DIR}/{MODEL_NAME}_optuna.db'
STUDY_NAME = f'{MODEL_NAME}_stage1'

study = run_study(
    objective  = objective,
    n_trials   = N_TRIALS,
    study_name = STUDY_NAME,
    direction  = 'maximize',
    storage    = DB_PATH,
)
print('Study complete.')

In [ ]:
# ── Report results ────────────────────────────────────────────────────────────
df_trials = report_study(study, top_k=10)

csv_path = SAVE_DIR / f'{MODEL_NAME}_stage1_trials.csv'
df_trials.to_csv(csv_path, index=False)
print(f'All trials saved → {csv_path}')

In [ ]:
# ── Optuna built-in visualisations ───────────────────────────────────────────
try:
    from optuna.visualization.matplotlib import (
        plot_optimization_history,
        plot_param_importances,
        plot_parallel_coordinate,
    )
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))

    plt.sca(axes[0])
    plot_optimization_history(study, target_name='val F1')
    axes[0].set_title('Optimisation History')

    plt.sca(axes[1])
    plot_param_importances(study)
    axes[1].set_title('HP Importances')

    plt.sca(axes[2])
    plot_parallel_coordinate(study)
    axes[2].set_title('Parallel Coordinate')

    plt.suptitle(f'{MODEL_NAME} — Stage 1 Optuna Search', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig_path = SAVE_DIR / f'{MODEL_NAME}_stage1_plots.png'
    plt.savefig(fig_path, dpi=120, bbox_inches='tight')
    print(f'Plot saved → {fig_path}')
    plt.show()
except Exception as e:
    print(f'Matplotlib visualisation skipped ({e}).\n'
          'Run optuna.visualization.plot_* for interactive Plotly charts.')

In [ ]:
# ── Save best HP for Stage 2 ──────────────────────────────────────────────────
best = study.best_trial
best_hp = {
    'model_name': MODEL_NAME,
    'best_val_f1': best.value,
    'trial_number': best.number,
    **best.params,
}

hp_path = SAVE_DIR / f'{MODEL_NAME}_best_hp.json'
with open(hp_path, 'w') as f:
    json.dump(best_hp, f, indent=2)
print(f'Best HP saved → {hp_path}')
print(json.dumps(best_hp, indent=2))